# 04 · Table 1: baseline characteristics by SAA status

Baseline characteristics of SAA-positive vs SAA-negative prodromal participants, with baseline blood chemistry (serum uric acid, glucose, creatinine), the four prodromal subgroups and the CSF SAA assay. Continuous variables: Mann–Whitney U; categorical: χ², Fisher's exact or permutation χ² for sparse tables; Holm correction across all characteristics.

**Inputs:** `results_baseline/prodromal_saa_longitudinal.csv` (notebook 01) and `data/Blood_Chemistry___Hematology_13Sep2026.csv` (PPMI)

**Outputs** (`analysis_by_group/table1_with_subgroups/`): `table1_SAA_with_subgroups.csv`, `table1_SAA_with_subgroups.md`, `level_check.txt`

In [ ]:
# Works in Google Colab (mounts Drive) and locally. Set the project folder here,
# or through the PPMI_SAA_BASE environment variable.
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

BASE_DIR = os.environ.get('PPMI_SAA_BASE',
                          '/content/drive/MyDrive/PPMI_Projects/prodromal_SAA_groups')

## Configuration

In [ ]:
# ------------------------------- CONFIG -------------------------------------
T1_LAB_FILE     = 'Blood_Chemistry___Hematology_13Sep2026.csv'   # in <BASE_DIR>/data/
T1_OUT_NAME     = 'analysis_by_group/table1_with_subgroups'     # under <BASE_DIR>/

T1_CONT_SUMMARY = 'mean'       # 'mean' (mean ± SD) or 'median' (median [Q1–Q3])
T1_MT           = 'holm'       # multiple-testing correction across all characteristics
T1_ALPHA        = 0.05
T1_PERM_N       = 20000        # permutations for sparse tables larger than 2x2
T1_SEED         = 0

# Laboratory thresholds (SI units, baseline visit only)
T1_URIC_LIMIT    = 404         # µmol/L; hyperuricaemia = value >= limit
T1_GLUCOSE_LIMIT = 7.0         # mmol/L; elevated glucose = value >= limit
                               # creatinine: elevated = value > the laboratory upper limit (LSIHIRNG)

T1_SUBGROUP_ORDER = [
    ('A2', 'PSG-RBD only (A2)'),
    ('C',  'Hyposmia only (C)'),
    ('A1', 'Hyposmia + PSG-RBD (A1)'),
    ('B',  'Hyposmia + possible RBD (B)'),
]
T1_REPORT_UNCLASSIFIED = True

T1_ASSAY_ORDER = [
    ('1', '150-hour pilot assay (Project 155)'),
    ('2', '24-hour assay (Project 237)'),
    ('3', '35-hour assay (Project 299)'),
]
T1_REPORT_ASSAY_MISSING = True

# (column, row label, type): 'cont' = continuous, 'bin' = binary, 'cat' = categorical
T1_VARS = [
    ('age_at_visit',        'Age at baseline, years',                       'cont'),
    ('SEX',                 'Male sex, n (%)',                              'bin'),
    ('EDUCYRS',             'Education, years',                             'cont'),
    ('BMI',                 'Body mass index, kg/m²',                       'cont'),
    ('hyperuricaemia',      'Serum hyperuricaemia (≥404 µmol/L), n (%)',     'bin'),
    ('elevated_glucose',    'Elevated serum glucose (≥7.0 mmol/L), n (%)',   'bin'),
    ('elevated_creatinine', 'Elevated serum creatinine (above laboratory-specific upper limit), n (%)', 'bin'),
    ('race',                'White race, n (%)',                            'bin'),
    ('fampd_bin',           "Family history of Parkinson's disease, n (%)", 'bin'),
    ('APOE_e4',             'APOE ε4 allele count, n (%)',                  'cat'),
    ('moca',                'MoCA score',                                   'cont'),
    ('hy',                  'Hoehn and Yahr stage',                         'cont'),
    ('pigd',                'PIGD score',                                   'cont'),
    ('updrs3_score',        'MDS-UPDRS Part III score',                     'cont'),
    ('__subgroup__',        'Subgroups, n (%)',                             'cat'),
    ('CSFSAA_assay',        'CSF SAA assay, n (%)',                         'cat'),
]

# Values counted as "yes" for binary variables (compared after upper-casing)
T1_POSITIVE_LEVEL = {
    'SEX':                 ['1', 'M', 'MALE'],
    'race':                ['1', 'WHITE', 'W', 'CAUCASIAN'],
    'fampd_bin':           ['1', 'Y', 'YES', 'TRUE'],
    'hyperuricaemia':      ['1', 'TRUE'],
    'elevated_glucose':    ['1', 'TRUE'],
    'elevated_creatinine': ['1', 'TRUE'],
}
# ---------------------------- END OF CONFIG ---------------------------------

## Build Table 1

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

T1_BASE = Path(BASE_DIR)
T1_SRC  = T1_BASE / 'results_baseline' / 'prodromal_saa_longitudinal.csv'
T1_LAB  = T1_BASE / 'data' / T1_LAB_FILE
T1_OUT  = T1_BASE / T1_OUT_NAME
T1_OUT.mkdir(parents=True, exist_ok=True)

if not T1_SRC.exists():
    raise FileNotFoundError(f'Missing input: {T1_SRC}  (run 01_build_dataset first)')
if not T1_LAB.exists():
    raise FileNotFoundError(f'Missing blood chemistry file: {T1_LAB}')

t1_rng = np.random.default_rng(T1_SEED)


# ============================================================================
# Functions
# ============================================================================
def t1_norm(v):
    s = str(v).strip().upper()
    return s[:-2] if s.endswith('.0') else s


def t1_fmt_p(p):
    if p is None or (isinstance(p, float) and not np.isfinite(p)):
        return ''
    return '<0.001' if p < 0.001 else f'{p:.3f}'


def t1_holm(pv):
    p = np.asarray(pv, dtype=float)
    ok = np.isfinite(p)
    out = np.full_like(p, np.nan)
    if ok.sum():
        out[ok] = multipletests(p[ok], method=T1_MT)[1]
    return out


def t1_binary(series, col):
    """True / False per participant (NA stays NA), using T1_POSITIVE_LEVEL."""
    levels = [t1_norm(x) for x in T1_POSITIVE_LEVEL.get(col, ['1'])]
    b = pd.Series(pd.NA, index=series.index, dtype='object')
    ok = series.notna()
    b[ok] = series[ok].map(lambda v: t1_norm(v) in levels)
    return b


def t1_perm_chi2(tab):
    """Permutation chi-square test for sparse tables larger than 2x2."""
    tab = np.asarray(tab, dtype=float)
    obs = stats.chi2_contingency(tab, correction=False)[0]
    rows, cols = tab.shape
    vals = np.repeat(np.arange(rows), tab.sum(axis=1).astype(int))
    grps = np.concatenate([np.full(int(n), j) for j, n in enumerate(tab.sum(axis=0))])
    hit = 0
    for _ in range(T1_PERM_N):
        g = t1_rng.permutation(grps)
        t = np.zeros((rows, cols))
        np.add.at(t, (vals, g), 1)
        keep = t.sum(axis=1) > 0
        try:
            st = stats.chi2_contingency(t[keep], correction=False)[0]
        except Exception:
            continue
        hit += st >= obs - 1e-9
    return (hit + 1) / (T1_PERM_N + 1), f'permutation χ² ({T1_PERM_N:,})'


def t1_cat_test(tab):
    """χ² (continuity corrected for 2x2); Fisher's exact for sparse 2x2;
    permutation χ² for sparse larger tables."""
    tab = np.asarray(tab, dtype=float)
    tab = tab[tab.sum(axis=1) > 0][:, tab.sum(axis=0) > 0]
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan, 'n/a'
    try:
        chi2, p, dof, exp = stats.chi2_contingency(tab, correction=(tab.shape == (2, 2)))
    except Exception:
        return np.nan, 'n/a'
    if (exp < 5).any():
        if tab.shape == (2, 2):
            return stats.fisher_exact(tab)[1], "Fisher's exact ‡"
        return t1_perm_chi2(tab)
    return p, 'χ²' + (' (continuity corrected)' if tab.shape == (2, 2) else '')


# ============================================================================
# Baseline dataset: one row per participant
# ============================================================================
d = pd.read_csv(T1_SRC, na_values=[''], low_memory=False).sort_values(['PATNO', 'visit_n'])
d['subgroup_lab'] = d.analytic_subgroup.fillna('Missing')
bl = d[d.is_baseline == 1].set_index('PATNO').copy()
bl['__subgroup__'] = bl.subgroup_lab


# ============================================================================
# Blood chemistry -- baseline (BL) measurements only
# ============================================================================
lab = pd.read_csv(T1_LAB, low_memory=False)
for _c in ('PATNO', 'LSIRES', 'LSIHIRNG'):
    lab[_c] = pd.to_numeric(lab[_c], errors='coerce')
lab_bl = lab[lab['EVENT_ID'].astype(str).str.strip().str.upper().eq('BL')].copy()


def t1_lab_test(name, regex=False, need_upper=False):
    """One BL value per participant for a test (first by PATNO, LCOLLDT)."""
    names = lab_bl['LTSTNAME'].astype(str)
    sel = (names.str.contains(name, case=False, na=False, regex=True) if regex
           else names.str.strip().eq(name))
    need = ['PATNO', 'LSIRES'] + (['LSIHIRNG'] if need_upper else [])
    return (lab_bl[sel].copy()
            .dropna(subset=need)
            .sort_values(['PATNO', 'LCOLLDT'])
            .drop_duplicates('PATNO', keep='first')
            .set_index('PATNO'))


def t1_set_flag(col, mask, values):
    """Object column: True/False where the lab value is available, NA otherwise."""
    bl[col] = pd.Series(pd.NA, index=bl.index, dtype='object')
    bl.loc[mask, col] = values


# Serum uric acid: hyperuricaemia = value >= T1_URIC_LIMIT µmol/L
uric = t1_lab_test('Serum Uric Acid')
uric_value = uric['LSIRES'].reindex(bl.index)
mask = uric_value.notna()
t1_set_flag('hyperuricaemia', mask, uric_value.loc[mask] >= T1_URIC_LIMIT)

# Serum glucose: elevated = value >= T1_GLUCOSE_LIMIT mmol/L
glucose = t1_lab_test('Serum Glucose')
glucose_value = glucose['LSIRES'].reindex(bl.index)
mask = glucose_value.notna()
t1_set_flag('elevated_glucose', mask, glucose_value.loc[mask] >= T1_GLUCOSE_LIMIT)

# Serum creatinine: elevated = value > laboratory-specific upper limit (LSIHIRNG)
creatinine = t1_lab_test(r'Creatinine\s*\(Rate Blanked\)', regex=True, need_upper=True)
creatinine_value = creatinine['LSIRES'].reindex(bl.index)
creatinine_upper = creatinine['LSIHIRNG'].reindex(bl.index)
mask = creatinine_value.notna() & creatinine_upper.notna()
t1_set_flag('elevated_creatinine', mask, creatinine_value.loc[mask] > creatinine_upper.loc[mask])

print('Serum creatinine available:', bl['elevated_creatinine'].notna().sum())
print('\n=== BL blood chemistry availability ===')
print('BL laboratory rows:', len(lab_bl))
print('Serum uric acid:', bl['hyperuricaemia'].notna().sum())
print('Serum glucose:', bl['elevated_glucose'].notna().sum())
print('Serum creatinine + LSIHIRNG:', bl['elevated_creatinine'].notna().sum())


# ============================================================================
# SAA groups
# ============================================================================
ARMS = [('Positive', 'SAA-positive'), ('Negative', 'SAA-negative')]
KEYS = [k for k, _ in ARMS]
arm = {k: bl[bl.SAA_Status_BL == k] for k in KEYS}
ntot = {k: len(arm[k]) for k in KEYS}
n_other = len(bl) - sum(ntot.values())
print(f'Baseline n = {len(bl)}   SAA+ = {ntot["Positive"]}   SAA- = {ntot["Negative"]}'
      + (f'   (excluded, indeterminate/missing SAA = {n_other})' if n_other else ''))


# ============================================================================
# Level-detection report: which raw values are counted as "yes"
# ============================================================================
chk = ['Level check (all baseline participants)', '']
for col, label, kind in T1_VARS:
    if col not in bl.columns:
        chk.append(f'{col}: COLUMN NOT FOUND')
        continue
    if kind in ('bin', 'cat'):
        chk.append(f'{col}  ({label})')
        yes = [t1_norm(x) for x in T1_POSITIVE_LEVEL.get(col, ['1'])]
        for k, v in bl[col].value_counts(dropna=False).items():
            mark = '   <-- counted' if (kind == 'bin' and t1_norm(k) in yes) else ''
            chk.append(f'    {k!r}: {v}{mark}')
        chk.append('')
(T1_OUT / 'level_check.txt').write_text('\n'.join(chk), encoding='utf-8')
print('\n'.join(chk))


# ============================================================================
# Table 1 rows
# ============================================================================
rows, raw_p, tests = [], [], []


def t1_push(label, cells, p=np.nan, test=''):
    rows.append({'characteristic': label, **cells})
    raw_p.append(p)
    tests.append(test)


def t1_n_pct(c, n):
    return f'{c} ({100*c/n:.1f})' if n else 'n/a'


for col, label, kind in T1_VARS:
    if col not in bl.columns:
        continue

    # ---------------- continuous: mean ± SD (or median [IQR]), Mann-Whitney U
    if kind == 'cont':
        cells, samples = {}, {}
        for k in KEYS:
            x = pd.to_numeric(arm[k][col], errors='coerce').dropna()
            samples[k] = x.values
            if not len(x):
                cells[k] = 'n/a'
                continue
            if T1_CONT_SUMMARY == 'median':
                body = f'{x.median():.2f} [{x.quantile(.25):.2f}–{x.quantile(.75):.2f}]'
            else:
                body = f'{x.mean():.2f} ± {x.std():.2f}'
            cells[k] = body + (f' [n={len(x)}]' if len(x) != ntot[k] else '')
        p = (stats.mannwhitneyu(samples[KEYS[0]], samples[KEYS[1]], alternative='two-sided').pvalue
             if min(len(samples[KEYS[0]]), len(samples[KEYS[1]])) >= 3 else np.nan)
        t1_push(label, cells, p, 'Mann-Whitney U')

    # ---------------- binary: n (%)
    elif kind == 'bin':
        b = {k: t1_binary(arm[k][col], col).dropna().astype(bool) for k in KEYS}
        cells = {}
        for k in KEYS:
            n, c = len(b[k]), int(b[k].sum())
            cells[k] = (t1_n_pct(c, n) + (f' [n={n}]' if n != ntot[k] else '')) if n else 'n/a'
        tab = np.array([[int((~b[k]).sum()) for k in KEYS],
                        [int(b[k].sum()) for k in KEYS]])
        p, tname = t1_cat_test(tab)
        t1_push(label, cells, p, tname)

    # ---------------- categorical: header row with the test, then one row per level
    else:
        s = {k: arm[k][col].dropna().map(t1_norm) for k in KEYS}
        if col == '__subgroup__':
            levels = [t1_norm(g) for g, _ in T1_SUBGROUP_ORDER]
            lab_map = {t1_norm(g): lb for g, lb in T1_SUBGROUP_ORDER}
            s = {k: v[v.isin(levels)] for k, v in s.items()}
        elif col == 'CSFSAA_assay':
            lab_map = {t1_norm(c): lb for c, lb in T1_ASSAY_ORDER}
            levels = [t1_norm(c) for c, _ in T1_ASSAY_ORDER]
            levels += sorted({v for k in KEYS for v in s[k]} - set(levels))
            s = {k: v[v.isin(levels)] for k, v in s.items()}
        else:
            levels = sorted({v for k in KEYS for v in s[k]}, key=lambda x: (not x.isdigit(), x))
            lab_map = {'0': '0 copies', '1': '1 copy', '2': '2 copies'}

        tab = np.array([[int((s[k] == l).sum()) for k in KEYS] for l in levels])
        p, tname = t1_cat_test(tab)
        t1_push(label, {k: f'n = {len(s[k])}' for k in KEYS}, p, tname)

        for l in levels:
            t1_push('    ' + lab_map.get(l, l),
                    {k: t1_n_pct(int((s[k] == l).sum()), len(s[k])) for k in KEYS})

        if col == 'APOE_e4':
            carr = [l for l in levels if l.isdigit() and int(l) >= 1]
            t1_push('    ε4 carriers (≥1 allele)',
                    {k: t1_n_pct(int(s[k].isin(carr).sum()), len(s[k])) for k in KEYS})

        if col == '__subgroup__' and T1_REPORT_UNCLASSIFIED:
            t1_push('    Unclassified (not tested)',
                    {k: str(int((arm[k]['__subgroup__'] == 'Missing').sum())) for k in KEYS})

        if col == 'CSFSAA_assay' and T1_REPORT_ASSAY_MISSING:
            t1_push('    Not recorded (not tested)',
                    {k: str(int(ntot[k] - len(s[k]))) for k in KEYS})


# ============================================================================
# Holm correction and CSV
# ============================================================================
p_adj = t1_holm(raw_p)
n_fam = int(np.isfinite(raw_p).sum())

tbl = pd.DataFrame(rows)
tbl['P'] = [t1_fmt_p(x) for x in raw_p]
tbl['P adj.'] = [t1_fmt_p(x) for x in p_adj]
tbl['test'] = tests
tbl['p_raw_numeric'] = raw_p
tbl['p_adj_numeric'] = p_adj
tbl['sig_adj'] = [bool(np.isfinite(x) and x < T1_ALPHA) for x in p_adj]

t1_cols = [f'{lb} (n = {ntot[k]})' for k, lb in ARMS]
tbl = tbl.rename(columns=dict(zip(KEYS, t1_cols)))
tbl = tbl[['characteristic'] + t1_cols
          + ['P', 'P adj.', 'sig_adj', 'test', 'p_raw_numeric', 'p_adj_numeric']]
tbl.to_csv(T1_OUT / 'table1_SAA_with_subgroups.csv', index=False)


# ============================================================================
# Markdown copy with footnotes
# ============================================================================
hdr = ['Characteristic'] + [f'{lb}<br>(n = {ntot[k]})' for k, lb in ARMS] + ['P', 'P adj.']
md = ['| ' + ' | '.join(hdr) + ' |', '|' + '---|' * len(hdr)]
for _, r in tbl.iterrows():
    padj = f"**{r['P adj.']}**" if r.sig_adj else r['P adj.']
    md.append('| ' + ' | '.join([str(r.characteristic)] + [str(r[c]) for c in t1_cols]
                                + [r['P'], padj]) + ' |')

n_sub_pos = int((arm['Positive']['__subgroup__'] != 'Missing').sum())
n_sub_neg = int((arm['Negative']['__subgroup__'] != 'Missing').sum())
has_assay = 'CSFSAA_assay' in bl.columns
n_assay_pos = int(arm['Positive']['CSFSAA_assay'].notna().sum()) if has_assay else 0
n_assay_neg = int(arm['Negative']['CSFSAA_assay'].notna().sum()) if has_assay else 0

md += [
    '',
    ('Data are median [Q1–Q3]' if T1_CONT_SUMMARY == 'median' else 'Data are mean ± SD')
    + ' for continuous variables and n (%) for categorical variables. '
    'Percentages are calculated among participants with non-missing data; '
    'where the number with available data differs from the group total, '
    'it is given in brackets or in the variable header row.',
    '',
    'Race was dichotomised as White versus non-White. Continuous variables '
    'were compared with the Mann–Whitney U test; categorical variables with '
    'the χ² test (with continuity correction for 2×2 tables), Fisher\'s '
    'exact test (‡) where any expected cell count was <5, or a permutation '
    f'χ² test ({T1_PERM_N:,} permutations) for larger sparse tables.',
    '',
    'The Subgroups row compares the distribution of the four prodromal '
    'subgroups between SAA-positive and SAA-negative participants '
    f'({n_sub_pos} and {n_sub_neg} participants with an assigned subgroup, '
    'respectively). Participants who could not be assigned to one of the '
    'four subgroups are listed for completeness and were excluded from '
    'this comparison.',
    '',
    'The CSF SAA assay row gives the assay that produced each participant’s '
    'reported CSF SAA result (the curated result prioritises the most recent '
    f'assay available), for the {n_assay_pos} SAA-positive and '
    f'{n_assay_neg} SAA-negative participants with a recorded assay. '
    'Participants without a recorded assay are listed for completeness and '
    'were excluded from this comparison. This row describes assay provenance '
    'rather than a participant characteristic and is reported to allow '
    'assessment of assay-related heterogeneity.',
    '',
    f'P adj. = P value adjusted for multiple comparisons using the '
    f'Holm–Bonferroni procedure across the {n_fam} baseline characteristics; '
    f'values <{T1_ALPHA} are shown in bold.',
    '',
    'Abbreviations: SAA, seed amplification assay; RBD, REM sleep behaviour '
    'disorder; PSG, polysomnography; BMI, body-mass index; APOE, '
    'apolipoprotein E.',
]
(T1_OUT / 'table1_SAA_with_subgroups.md').write_text('\n'.join(md), encoding='utf-8')

print('\n=== Table 1 with Subgroups block (Holm family size = %d)' % n_fam)
print(tbl[['characteristic'] + t1_cols + ['P', 'P adj.']].to_string(index=False))
print('\n--- written to', T1_OUT)

## Check: creatinine upper limit by sex

In [ ]:
# Is the creatinine upper limit (LSIHIRNG) sex-specific?
# (It is, which is why elevated creatinine is defined against each participant's own limit.)
check_creat = creatinine.reset_index()[['PATNO', 'LSIRES', 'LSIHIRNG']].merge(
    bl[['SEX']].reset_index(), on='PATNO', how='left')

print('\n=== LSIHIRNG by sex ===')
print(pd.crosstab(check_creat['SEX'], check_creat['LSIHIRNG'], margins=True))

print('\nUnique upper limits by sex:')
for sex, x in check_creat.groupby('SEX'):
    print(sex, sorted(x['LSIHIRNG'].dropna().unique()))